In [ ]:
# Challenge 1 Group #7: Regression Problem - Aprtment Rental Price Prediction
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

In [ ]:
# loading datasets
train = pd.read_csv("challenge1_train.csv")
test = pd.read_csv("challenge1_test.csv")

train.head()

In [ ]:
X = train.drop(columns=["price", "ID"]).copy()
y = train["price"].copy()

# seperating training IDs
train_ids = train["ID"].copy()

# preparing test inputs and preserving IDs
X_test = test.drop(columns=["ID"]).copy()
test_ids = test["ID"].copy()

X.head()

In [ ]:
# Pipelines for Numerical and Categorical columns
numeric_features = [
    "bathrooms",
    "bedrooms",
    "square_feet",
    "latitude",
    "longitude"
]

categorical_features = [
    "category",
    "has_photo",
    "state"
]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        drop="first",
        sparse_output=False
    ))
])



In [ ]:
# Connecting Pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

preprocessor

In [ ]:
# checking for every predictor
assert set(numeric_features + categorical_features) == set(X.columns)

assert X.columns.equals(X_test.columns)

print("Preprocessing setup is ready")
print("Training inputs:", X.shape)
print("Training target:", y.shape)
print("Test inputs:", X_test.shape)

## Validation Strategy

In [ ]:
## continued imports

import numpy as np
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error

In [ ]:
## seed & safety check

RANDOM_STATE = 42
assert "ID" not in X.columns and "price" not in X.columns

In [ ]:
## hold out split

price_bins = pd.qcut(y, q=10, duplicates="drop")

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=price_bins
)

print("Train:", X_train.shape, "| Hold-out:", X_val.shape)

In [ ]:
## checking the split

pd.DataFrame({"train": y_train.describe(), "hold-out": y_val.describe()})

In [ ]:
## folds and scoring

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae":  "neg_mean_absolute_error",
    "r2":   "r2",
}

print([(len(tr), len(va)) for tr, va in cv.split(X_train)])

In [ ]:
## defining functions

def cv_evaluate(name, model):
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    res = cross_validate(pipe, X_train, y_train, cv=cv,
                         scoring=scoring, return_train_score=True)
    cv_rmse = -res["test_rmse"].mean()
    train_rmse = -res["train_rmse"].mean()
    return {
        "model": name,
        "CV RMSE (mean)": cv_rmse,
        "CV RMSE (std)": res["test_rmse"].std(),
        "Train RMSE": train_rmse,
        "Gap (CV - Train)": cv_rmse - train_rmse,
        "CV MAE": -res["test_mae"].mean(),
        "CV R2": res["test_r2"].mean(),
    }

def compare(models):
    rows = [cv_evaluate(n, m) for n, m in models.items()]
    return pd.DataFrame(rows).sort_values("CV RMSE (mean)").reset_index(drop=True)

def holdout_rmse(model):
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    return np.sqrt(mean_squared_error(y_val, pipe.predict(X_val)))

In [ ]:
## final baseline checks

baseline = compare({"Baseline (mean price)": DummyRegressor(strategy="mean")})
print("Std of price in X_train:", y_train.std())
baseline

### How to use this validation setup
- Evaluate models with `compare({"Name": model, ...})`; give every model `random_state=RANDOM_STATE`.

- Grid search: `GridSearchCV(Pipeline([("prep", preprocessor), ("model", ...)]), param_grid={"model__<param>": [...]}, cv=cv, scoring="neg_root_mean_squared_error")`, fit on `X_train, y_train` only.
  
- Use `X_val` / `y_val` only once, via `holdout_rmse(final_model)`, after the final model is chosen.
  
- Before predicting `X_test`, refit the final model on all of `X, y`.